In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    seed = 43
    environment_string = "mini_grid"
    gold_timesteps = 1_000_000
    training_timesteps = 1 
    num_concepts_selected = 3
    out_folder = "basic"
    method = "lp" 


In [6]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [6]:
concept_list,processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
num_concepts_selected = min(num_concepts_selected,len(concept_list))
ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
if os.path.exists(model_name):
    groundtruth_model = PPO.load(model_name)
else:
    if "cyclic" in environment_string or "tree" in environment_string or "glucose" in environment_string:
        policy = "MlpPolicy"
    else:
        policy = "CnnPolicy"

    if policy == "MlpPolicy":
        groundtruth_model = train_ppo_model(ground_truth_env,environment_string+"_raw",total_timesteps=gold_timesteps,policy=policy)
    else:
        groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=gold_timesteps,policy=policy)
    
    groundtruth_model.save(model_name)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▂▁▁▁▁▁▁▁▁▄▄▄▆▆▆▆▅▅▅▅▇▇▆▆▆▆▆▅▅▅▅████▅▅▅▅▅
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▄▄▄▄▄▄▃▃▃▃▃▃▃▃██▄▄▄▄▄▅▅▅
ema_norm_reward,▃▂▂▂▂▂▂▁▁▁▁▂▆▄▂▅█▅▃▂▂▁▁▅▄▂▂▂▁▁▁▁▁▁▁▁▁▄▂▁
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▄▄▄▇▇▆▆▆████▇▇▇██▆▆▆███
episode_length_mean,▂████████████▃██████████████▁███████████
episode_reward_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁
episode_reward_min,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episodes_completed,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
explained_variance,▁▁▁█████▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▅▅▅▂▂▂▅▅▅▅▅▅▃▃▅▅▅
+1,...


In [8]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [13]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,concept_idx=list(range(len(concept_list))),processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=25_000,custom_name="{}_perfect_{}_{}".format(environment_string,method,seed)) 

approx_kl,██████████████▁▁▁▁▁▁▁▁▁▁▁▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,▁▂▂▂▂▂▁▄▆▅▂▂▂▂▁▁▄▃▆▃▃▃▂▂▂▂▂▂▆▆▅▇█▇▆▅▄▄▃▄
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄███████████
episode_length_mean,█▆██▇█████████████▃██████████▁▂█▆███████
episode_reward_max,▁▅▁▁▃▁▃▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▅▁▁▁▁▁▁
episode_reward_mean,▃▁▁▂▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁▂▁▁█▃▁▇▁▃▁▁▁▁▁
episode_reward_min,▄▁▁▂▁▁▁▆▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁▁▁▁▄▁█▁▁▁▁▁▁▁▁█▁▁
episodes_completed,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███
explained_variance,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁██████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
+1,...


In [22]:
evaluate_model(environment_string,two_stage_gym_env,model,seed,max_steps=50_000)

0.05426213590120806

In [17]:
np.max(two_stage_env.reset())

0.0017670575

In [9]:
groundtruth_model.predict(ground_truth_gym_env.reset()[0])

(array([0, 0, 3, 0, 4, 0, 3, 1]), None)

In [10]:
q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)

Starting stable training for sparse rewards...
Total of 200000 steps
Step 0/200000, Loss mean: 0.0000


ValueError: Error: Unexpected observation shape (8, 1, 84, 84) for Box environment, please use (4, 84, 84) or (n_env, 4, 84, 84) for the observation shape.

In [41]:

if is_main:
    model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

    height = width = 84

    if environment_string == "mini_grid":
        num_frames = 1
    else:
        num_frames = 4

    if environment_string == "cart_pole":
        height = 160
        width = 240

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if os.path.exists(model_name):
        concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
        concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
        concept_predictor.eval()

In [42]:
idx = [6,7,9]

In [45]:
wandb.finish()

approx_kl,▅▅▅▅▅▅▅▅▅▅███████████████▃▃▃▃▃▃▃▃▃▃▃▃▁▁▁
clip_fraction,██████████▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▄▄▄▄
ema_norm_reward,▁▂▃▄▄▃▄▄▃▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▆▆█▆▇▆▆▅▆▆▇██▇▇
entropy_loss,▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▄▄▄▄▄▄▄▄▄▄▄▄▄▄█████
episode_length_mean,▁▃▂▁▄▁▄▂▁▃▃▂▂▁▂▂▂▂▂▁▆▃▄█▄▃▂▂▂▅▄▄▄▃▁▄▆▄▆▂
episode_reward_max,▁▁▂▂▃▂▁▁▂▁▃▂▂▂▂▃▃▄▂▂▂▂▃▂▄█▂▃▄▂▃▃▁▃▅▂▇▃▂▄
episode_reward_mean,▁▁▂▂▂▂▂▁▂▂▁▂▃▂▁▂▂▂▁▂▂▇▄▆▃▄▂▂▄▂▄▁▃▅▇▄█▆▅▅
episode_reward_min,▁▁▁▂▁▂▁▁▁▁▂▂▁▃▁▁▁▁▁▁▁▁▁▃▂▁▁▁▁▃▃▃▂▄▃▄▃█▅▁
episodes_completed,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇██
explained_variance,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁███████████████████████▆▆
+1,...


In [56]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=0.5,processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_testing".format(environment_string))    

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

KeyboardInterrupt: 

In [82]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=list(range(len(concept_list))),fast_predictor=concept_predictor,use_processed=True)
two_stage_env

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [38]:
import matplotlib.pyplot as plt 
ground_truth_env.reset()
for i in range(10):
    obs, _, _, _ = ground_truth_env.step([i%4 for i in range(8)])

torch.max(torch.abs(concept_predictor(torch.tensor(obs).float().cuda())))

tensor(4442.1084, device='cuda:0', grad_fn=<MaxBackward1>)

In [95]:
for i in range(100):
    print(np.max(np.std(two_stage_gym_env.step([i for i in range(8)])[0],axis=0)))

Sigmoid predictions tensor([[-12.3487, -12.3487, -12.3487,  ...,  13.9105,   9.8699,   8.6226],
        [-11.3817, -11.3817, -11.3817,  ...,  13.0597,   8.5640,   6.8425],
        [-12.3487, -12.3487, -12.3487,  ...,  13.9105,   9.8699,   8.6226],
        ...,
        [-12.3487, -12.3487, -12.3487,  ...,  13.9105,   9.8699,   8.6226],
        [-12.3487, -12.3487, -12.3487,  ...,  13.9105,   9.8699,   8.6226],
        [-12.3487, -12.3487, -12.3487,  ...,  13.9105,   9.8699,   8.6226]],
       device='cuda:0')
0.7399304
Sigmoid predictions tensor([[-11.3370, -11.3370, -11.3370,  ...,  13.9770,   8.5687,   6.3190],
        [-11.4813, -11.4813, -11.4813,  ...,  13.6480,   8.5094,   6.7396],
        [-11.4857, -11.4857, -11.4857,  ...,  15.7185,   8.8634,   6.4360],
        ...,
        [-11.4620, -11.4620, -11.4620,  ...,  13.2360,   8.1475,   6.1143],
        [-11.7814, -11.7814, -11.7814,  ...,  15.8253,   9.1604,   7.1397],
        [-11.4857, -11.4857, -11.4857,  ...,  15.7185,   8.8634

In [56]:
two_stage_env.reset()

Sigmoid predictions tensor([[0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727],
        [0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727],
        [0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727],
        ...,
        [0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727],
        [0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727],
        [0.0011, 0.0011, 0.0011,  ..., 0.8893, 0.9951, 0.0727]],
       device='cuda:0')


array([[0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623],
       [0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623],
       [0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623],
       ...,
       [0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623],
       [0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623],
       [0.00111532, 0.00111532, 0.00111532, ..., 0.88933265, 0.99505496,
        0.07267623]], dtype=float32)

In [11]:
subset_concepts, idx = policy_coverage_selection_lp_hybrid(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates)

6
There are 8000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
There are 51.0 x vals
6
There are 8000 observations
There are 57.0 x vals
Coverage 0.9983


In [74]:
len(idx)

48

In [77]:
two_stage_gym_env.step([1 for i in range(8)])[-1]

Sigmoid predictions tensor([[-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565],
        [-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565],
        [-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565],
        ...,
        [-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565],
        [-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565],
        [-7.1887, -7.1887, -7.1887,  ...,  5.5983,  5.7708,  1.8565]],
       device='cuda:0')


({'lives': 0,
  'episode_frame_number': 2,
  'frame_number': 12,
  'observation': array([[ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 112, 124]]),
  'TimeLimit.truncated': False},
 {'lives': 0,
  'episode_frame_number': 2,
  'frame_number': 12,
  'observation': array([[ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 112, 124]]),
  'TimeLimit.truncated': False},
 {'lives': 0,
  'episode_frame_number': 2,
  'frame_number': 12,
  'observation': array([[ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 112, 124]]),
  'TimeLimit.truncated': False},
 {'lives': 0,
  'episode_frame_number': 2,
  'frame_number': 12,
  'observation': array([[ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 113, 125],
         [ 35,  42, 112, 124]]),
  'TimeLimit.truncated': False},
 {'lives': 0,
  'episode_frame_number': 2,
 

In [78]:
subset_concepts, idx_2 = policy_coverage_selection_lp_weighted(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates)

18
There are 8000 observations
There are 48.0 x vals
Coverage 0.96945


In [ ]:
len(idx_2) 

[14,
 35,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 67,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 86,
 110,
 111,
 112,
 117,
 120,
 121,
 122,
 123,
 129,
 131,
 132,
 133,
 135,
 174,
 176,
 189]

In [77]:
idx 

[0, 1, 2, 5, 6, 7]

In [78]:
policy_coverage_selection_lp(ground_truth_gym_env,concept_list,6,groundtruth_model)

There are 8000 observations
Coverage 0.9524


([<function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>],
 [0, 1, 6, 7, 8, 11])

In [51]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=[1,4,7])
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_{}_{}".format(environment_string,method,seed)) 

KeyboardInterrupt: 

In [40]:
sorted(final_vals,reverse=True)

[(0.47346973419189453, (0, 1, 3, 8, 9, 10, 11)),
 (0.47167205810546875, (5, 6, 7, 10)),
 (0.47156429290771484, (3, 4, 5, 6, 7, 11)),
 (0.4192380905151367, (0, 1, 4, 8)),
 (0.41866111755371094, (0, 1, 3, 8, 9)),
 (0.41733264923095703, (5, 6, 7, 9, 10)),
 (0.4136648178100586, (4, 8, 9, 10)),
 (0.41242027282714844, (0, 1, 3, 8, 9, 10)),
 (0.41051483154296875, (3, 4, 5, 6, 7)),
 (0.4000434875488281, (0, 1, 2, 3, 8, 9, 10, 11)),
 (0.39818477630615234, (4, 9, 10)),
 (0.39813804626464844, (2, 3, 4, 5, 6, 7, 11)),
 (0.3907032012939453, (4, 5, 6, 7)),
 (0.3823413848876953, (0, 2, 3, 4, 5, 10)),
 (0.3802204132080078, (1, 2, 3, 4, 5, 6, 7, 11)),
 (0.37216854095458984, (4, 5, 6, 7, 10)),
 (0.3612356185913086, (0, 1, 2, 3, 4, 10)),
 (0.35881519317626953, (0, 1, 2, 3, 4, 5, 9, 10)),
 (0.35809993743896484, (1, 3, 6, 7, 8, 9)),
 (0.3530111312866211, (0, 3, 4, 5, 10)),
 (0.35185909271240234, (1, 3, 6, 7, 8, 9, 10)),
 (0.3485536575317383, (1, 2, 3, 4, 5, 7, 10)),
 (0.3450336456298828, (2, 3, 4, 5, 10)),

In [32]:
q_estimates

[(array([1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0.]),
  0,
  11.440361022949219),
 (array([1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0.]),
  1,
  11.438813209533691),
 (array([1., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 1.]),
  0,
  11.548104286193848),
 (array([1., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 1.]),
  1,
  11.49608325958252),
 (array([0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]),
  0,
  11.353100776672363),
 (array([0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]),
  1,
  11.350688934326172),
 (array([1., 1., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.]),
  0,
  11.338913917541504),
 (array([1., 1., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.]),
  1,
  11.328780174255371),
 (array([1., 1., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1.]),
  0,
  11.429183959960938),
 (array([1., 1., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1.]),
  1,
  11.402504920959473),
 (array([0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 1., 1.]),
  0,
  11.270588874816895),
 (array([0., 0., 0., 0., 1., 1., 0., 1., 0.,

In [13]:
max_prefix_gurobi(final_vals,num_concepts_selected,in_order=False,weighted=True)

([1, 4, 7], 1537)

In [101]:
rng = np.random.default_rng()
num_pairs_lp=20_000
rollout_steps=1_000
# --------------------------------------------------
# Collect observations / actions (same as before)
# --------------------------------------------------
all_observations = []
all_actions = []

obs, info = ground_truth_gym_env.reset()
for _ in range(rollout_steps):
    actions = groundtruth_model.predict(obs)[0]
    for j in range(len(actions)):
        all_observations.append([c(info[j]['observation']) for c in concept_list])
        all_actions.append(actions[j])
    obs, rew, t_1, t_2, info = ground_truth_gym_env.step(actions)

all_observations = np.asarray(all_observations, dtype=np.int8)
all_actions = np.asarray(all_actions)

N, K = all_observations.shape

print("There are {} observations".format(N))

# --------------------------------------------------
# Sample cross-action pairs
# --------------------------------------------------
idx_i = rng.integers(0, N, size=5 * num_pairs_lp)
idx_j = rng.integers(0, N, size=5 * num_pairs_lp)

valid = all_actions[idx_i] != all_actions[idx_j]
idx_i = idx_i[valid][:num_pairs_lp]
idx_j = idx_j[valid][:num_pairs_lp]

if len(idx_i) == 0:
    raise ValueError("No cross-action pairs sampled.")

M = len(idx_i)

disagreement = (all_observations[idx_i] != all_observations[idx_j]).astype(np.int8)

# --------------------------------------------------
# Build LP in Gurobi
# --------------------------------------------------
model = gp.Model("max_coverage_lp")
model.Params.OutputFlag = 0

ub = 1.0
len_x_vals = 0
trials = 0

There are 8000 observations


In [92]:
acc_list = np.array([[1.0, 1.0, 1.0, 1.0, 0.9723301011417069, 0.942472000200446, 0.9930177979337359, 0.9836887073739069, 0.9942121219713863, 0.9798134181888034, 1.0, 1.0, 1.0, 1.0, 0.972614066297512, 0.942597278945654, 0.9930010941010415, 0.9837137631229486, 0.9942121219713863, 0.9797466028580257, 1.0, 1.0, 1.0, 1.0, 0.9725806586321232, 0.9429313555995423, 0.9929927421846942, 0.9836803554575597, 0.9942121219713863, 0.979796714356109, 1.0, 1.0, 1.0, 1.0, 0.972655825879248, 0.94243024061871, 0.9929760383519999, 0.9837054112066014, 0.9942121219713863, 0.9797883624397618, 1.0, 1.0, 1.0, 1.0, 0.9725890105484704, 0.9428812441014591, 0.9930261498500831, 0.9837471707883374, 0.9942121219713863, 0.9798050662724562, 1.0, 1.0, 1.0, 1.0, 0.9724553798869151, 0.9426891500254734, 0.9930512055991247, 0.9836887073739069, 0.9942121219713863, 0.9798050662724562, 1.0, 1.0, 1.0, 1.0, 0.9725639547994287, 0.942472000200446, 0.9930345017664303, 0.9836051882104349, 0.9942121219713863, 0.9796964913599425, 1.0, 1.0, 1.0, 1.0, 0.9725806586321232, 0.9426056308620013, 0.9930428536827776, 0.9836970592902542, 0.9942121219713863, 0.9798718816032339, 1.0, 1.0, 1.0, 1.0, 0.9728813276206225, 0.9423383695388907, 0.9930846132645136, 0.9837638746210318, 0.9942121219713863, 0.9796797875272482, 1.0, 1.0, 1.0, 1.0, 0.972856271871581, 0.9423717772042796, 0.9930094460173887, 0.9837221150392957, 0.9942121219713863, 0.9796547317782065, 1.0, 1.0, 1.0, 1.0, 0.9782683136645703, 0.9393233277375493, 0.9920656794701544, 0.984340156848989, 0.9942121219713863, 0.9817260070323136, 0.9890172300034242, 0.9878062021330795, 0.9984298397267253, 0.9916731394018358, 0.9780595157558902, 0.939306623904855, 0.992090735219196, 0.9843485087653362, 0.9942121219713863, 0.9817594146977023, 0.9926002021163756, 0.992775592359667, 0.9898941812198809, 0.9924999791202092, 0.9778423659308628, 0.9392398085740773, 0.9920656794701544, 0.9843735645143779, 0.9942121219713863, 0.9816007282871054, 0.9960913031495077, 0.9923747003750011, 0.9972939791035053, 0.9939699163973174, 0.9775834565240995, 0.9391646413269524, 0.992090735219196, 0.9843485087653362, 0.9942121219713863, 0.9815840244544111, 1.0, 1.0, 1.0, 1.0, 0.9780678676722374, 0.9394402545664102, 0.9920656794701544, 0.9843652125980307, 0.9942121219713863, 0.9816508397851887, 1.0, 1.0, 1.0, 1.0, 0.9781513868357095, 0.9393817911519798, 0.9920740313865016, 0.9843652125980307, 0.9942121219713863, 0.981742710865008, 1.0, 1.0, 1.0, 1.0, 0.9780678676722374, 0.939348383486591, 0.9920740313865016, 0.9844487317615027, 0.9942121219713863, 0.9817260070323136, 1.0, 1.0, 1.0, 1.0, 0.9778423659308628, 0.9391479374942581, 0.9920573275538073, 0.9844403798451554, 0.9942121219713863, 0.9816675436178831, 1.0, 1.0, 1.0, 1.0, 0.9779592927597237, 0.9390894740798276, 0.992090735219196, 0.9843318049326418, 0.9942121219713863, 0.981742710865008], [1.0, 1.0, 1.0, 1.0, 0.9775255599846556, 0.9254882665910569, 0.993853928648865, 0.97866804543256, 0.9999499641409677, 0.9775338992944944, 1.0, 1.0, 1.0, 1.0, 0.9775505779141718, 0.9256550527878313, 0.9938956251980586, 0.9785429557849792, 0.9999499641409677, 0.9775255599846556, 1.0, 1.0, 1.0, 1.0, 0.9774838634354621, 0.9253715162533149, 0.993853928648865, 0.9786180095735277, 0.9999499641409677, 0.9774922027453008, 1.0, 1.0, 1.0, 1.0, 0.9775005420551395, 0.9254966059008957, 0.9938622679587037, 0.9786597061227212, 0.9999499641409677, 0.9775088813649783, 1.0, 1.0, 1.0, 1.0, 0.9773671130977201, 0.9255549810697666, 0.9938205714095102, 0.9787847957703021, 0.9999499641409677, 0.9776589889420751, 1.0, 1.0, 1.0, 1.0, 0.9774671848157847, 0.9257301065763798, 0.993853928648865, 0.9784428840669146, 0.9999499641409677, 0.977458845505946, 1.0, 1.0, 1.0, 1.0, 0.9774338275764298, 0.9250546224794436, 0.9939039645078973, 0.9784512233767534, 0.9999499641409677, 0.9774004703370749, 1.0, 1.0, 1.0, 1.0, 0.9774838634354621, 0.9255799989992828, 0.9938872858882198, 0.9785846523341728, 0.9999499641409677, 0.9776423103223977, 1.0, 1.0, 1.0, 1.0, 0.9775422386043331, 0.925546641759928, 0.9938872858882198, 0.9784512233767534, 0.9999499641409677, 0.9776089530830429, 1.0, 1.0, 1.0, 1.0, 0.9774755241256233, 0.9253881948729923, 0.993853928648865, 0.9786180095735277, 0.9999499641409677, 0.9775505779141718, 1.0, 1.0, 1.0, 1.0, 0.9775088813649783, 0.9275814333605751, 0.9944710375769301, 0.9832630051536935, 0.9999499641409677, 0.9809446770185299, 0.9911269743316043, 0.9924195673566055, 0.9993745517620962, 0.9938289107193489, 0.977575595843688, 0.9274730223326717, 0.994446019647414, 0.9832963623930484, 0.9999499641409677, 0.9805944260053038, 0.9980402621879013, 0.9923528528778959, 0.9872825524959554, 0.9942792334506396, 0.9772420234501392, 0.9276564871491235, 0.9944543589572526, 0.9834297913504678, 0.9999499641409677, 0.9808195873709492, 1.0, 0.983246326534016, 0.9960471671364478, 0.9910269026135397, 0.9773504344780426, 0.9274563437129943, 0.994446019647414, 0.9832129692946612, 0.9999499641409677, 0.9807862301315943, 1.0, 1.0, 1.0, 1.0, 0.977517220674817, 0.9274980402621879, 0.9944626982670914, 0.983187951365145, 0.9999499641409677, 0.9808279266807879, 1.0, 1.0, 1.0, 1.0, 0.977575595843688, 0.9272478609670264, 0.9944210017178978, 0.9833130410127258, 0.9999499641409677, 0.9805777473856264, 1.0, 1.0, 1.0, 1.0, 0.9776256317027203, 0.9275313975015428, 0.9944793768867688, 0.9831796120553064, 0.9999499641409677, 0.9808029087512717, 1.0, 1.0, 1.0, 1.0, 0.9772920593091716, 0.9270560568407359, 0.9944543589572526, 0.9833380589422419, 0.9999499641409677, 0.9804860149774005, 1.0, 1.0, 1.0, 1.0, 0.9775922744633654, 0.9276314692196074, 0.9944710375769301, 0.9834214520406291, 0.9999499641409677, 0.9807278549627233], [1.0, 1.0, 1.0, 1.0, 0.9858337087650798, 0.9346832190352239, 0.9936176603093557, 0.9765563731624701, 0.9999415995060987, 0.9767148887887738, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9347833341676262, 0.9936593749478567, 0.9764646009577681, 0.9999415995060987, 0.9766898600056731, 1.0, 1.0, 1.0, 1.0, 0.9859254809697819, 0.934733276601425, 0.9936844037309572, 0.9764896297408686, 0.9999415995060987, 0.9767315746441741, 1.0, 1.0, 1.0, 1.0, 0.9859171380420817, 0.9345664180474212, 0.9936426890924563, 0.9766147736563715, 0.9999415995060987, 0.9767649463549749, 1.0, 1.0, 1.0, 1.0, 0.9858337087650798, 0.9347749912399259, 0.993626003237056, 0.9765146585239692, 0.9999415995060987, 0.9765647160901704, 1.0, 1.0, 1.0, 1.0, 0.9859421668251823, 0.9349585356493301, 0.9936844037309572, 0.9764062004638667, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9346581902521233, 0.9936009744539553, 0.9764228863192671, 0.9999415995060987, 0.976681517077973, 1.0, 1.0, 1.0, 1.0, 0.9859171380420817, 0.9346498473244231, 0.9936176603093557, 0.9765396873070697, 0.9999415995060987, 0.9767399175718743, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9347165907460246, 0.993626003237056, 0.9764896297408686, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9857836511988787, 0.9346748761075236, 0.9936510320201565, 0.9764812868131685, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9818457893243897, 0.9357761425639486, 0.9940848642605663, 0.9784418748227128, 0.9999582853614991, 0.9783667884734111, 0.9929085114548397, 0.9862425122223891, 0.9985983881463684, 0.9846573559593532, 0.9818624751797901, 0.9354507683836412, 0.9941599506098681, 0.9786170763044167, 0.9999582853614991, 0.9783501026180107, 0.9975722080392452, 0.9926415377684337, 0.9936009744539553, 0.9905724916987869, 0.9817873888304884, 0.9357177420700472, 0.9941349218267674, 0.9785336470274149, 0.9999582853614991, 0.9784168460396122, 1.0, 0.9939597203450635, 0.9975138075453438, 0.9943101233084715, 0.9818040746858888, 0.9359680299010529, 0.9940598354774658, 0.9785753616659159, 0.9999582853614991, 0.9784418748227128, 1.0, 1.0, 1.0, 1.0, 0.9818207605412892, 0.9357260849977475, 0.9941265788990673, 0.9785253040997147, 0.9999582853614991, 0.9784085031119121, 1.0, 1.0, 1.0, 1.0, 0.9819041898182911, 0.9357093991423471, 0.9940765213328662, 0.9785336470274149, 0.9999582853614991, 0.9784168460396122, 1.0, 1.0, 1.0, 1.0, 0.9818207605412892, 0.9357260849977475, 0.9941349218267674, 0.9785670187382156, 0.9999582853614991, 0.9783501026180107, 1.0, 1.0, 1.0, 1.0, 0.9817873888304884, 0.9358512289132502, 0.9941182359713671, 0.9785503328828152, 0.9999582853614991, 0.9784168460396122, 1.0, 1.0, 1.0, 1.0, 0.9819041898182911, 0.9357010562146468, 0.9941015501159667, 0.9784418748227128, 0.9999582853614991, 0.9784669036058133]][0])

In [93]:
subset_concepts, idx = policy_coverage_selection_lp(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

There are 8000 observations

Interrupt request received


RuntimeError: LP did not solve successfully

In [94]:
idx = [9, 14, 15, 16, 17, 24, 29, 39, 44, 46, 47, 56, 57, 65, 77, 85, 89, 94, 95, 96, 104, 107, 110, 111, 116, 119, 121, 122, 123, 124, 125, 127, 131, 132, 133, 137, 139, 145, 146, 149, 154, 155, 164, 165, 166, 169, 186, 187]

In [95]:
np.mean(np.array(acc_list)[idx])

0.977588328475302

In [89]:
np.mean(np.array(acc_list)[idx]) 

0.9648141356943444

In [96]:
subset_concepts, idx = policy_coverage_selection_exp_lp(
    ground_truth_gym_env,
    concept_list,
    acc_list,
    num_concepts_selected,
    groundtruth_model)

48.0


In [98]:
idx_multiple = idx 
idx_lp = [9, 14, 15, 16, 17, 24, 29, 39, 44, 46, 47, 56, 57, 65, 77, 85, 89, 94, 95, 96, 104, 107, 110, 111, 116, 119, 121, 122, 123, 124, 125, 127, 131, 132, 133, 137, 139, 145, 146, 149, 154, 155, 164, 165, 166, 169, 186, 187]

In [103]:
disagree_set = [set([j for j in range(len(i)) if i[j] == 1]) for i in disagreement]

In [116]:
len([i for i in disagree_set if i == set()])

316

In [121]:
[acc_list[i] for i in idx_multiple if i not in idx_lp]

[0.9930177979337359,
 0.9429313555995423,
 0.9929927421846942,
 0.9929760383519999,
 0.9837054112066014,
 0.9428812441014591,
 0.9426891500254734,
 0.9930345017664303,
 0.9930428536827776,
 0.9798718816032339,
 0.9728813276206225,
 0.9930846132645136,
 0.9837638746210318,
 0.9837221150392957,
 0.9920656794701544,
 0.9916731394018358,
 0.9920656794701544,
 0.992090735219196,
 0.9920740313865016,
 0.9844487317615027,
 0.9920573275538073,
 0.9844403798451554]

In [122]:
[acc_list[i] for i in idx_lp if i not in idx_multiple]

[0.972614066297512,
 0.942597278945654,
 0.9725806586321232,
 0.979796714356109,
 0.9797883624397618,
 0.9725890105484704,
 0.942472000200446,
 0.9423383695388907,
 0.9796797875272482,
 0.9423717772042796,
 0.984340156848989,
 0.992775592359667,
 0.9778423659308628,
 0.9392398085740773,
 0.9843735645143779,
 0.9923747003750011,
 0.9843485087653362,
 0.9815840244544111,
 0.9816508397851887,
 0.9780678676722374,
 0.9817260070323136,
 0.9843318049326418]

In [107]:
len(set(idx_lp).intersection(set(idx_multiple)))

26

In [108]:
np.mean([acc_list[i] for i in idx_lp if i not in idx_multiple])

0.9722492394061635

In [109]:
np.mean([acc_list[i] for i in idx_multiple if i not in idx_lp])

0.9823413914140783

In [97]:
np.mean(np.array(acc_list)[idx]) 

0.9822138981455962

0.9648141356943444

In [30]:
sorted(np.mean(disagreement,axis=0)), np.argsort(np.mean(disagreement,axis=0))

([0.18215,
  0.38235,
  0.45035,
  0.4514,
  0.4525,
  0.46185,
  0.4942,
  0.49495,
  0.4969,
  0.50055,
  0.5314,
  0.53265],
 array([ 1,  0,  5,  8, 11,  2,  9, 10,  4,  3,  7,  6]))

In [39]:
acc_list = [0.9767649687220733, 0.9794709896185679, 0.87484653353712, 0.8746377356284399, 0.8743370666399405, 0.8758654673314792, 0.9684381081239091, 0.9648133764292217, 0.8745124568832319, 0.8672713454102043, 0.8668788053418857, 0.8765920840536862]

In [40]:
subset_concepts, idx = policy_coverage_selection_exp_lp(
    ground_truth_gym_env,
    concept_list,
    acc_list,
    num_concepts_selected,
    groundtruth_model)


Interrupt request received
3.0


In [50]:
rng = np.random.default_rng()

# --------------------------------------------------
# Rollout data collection
# --------------------------------------------------
all_obs = []
all_actions = []

obs, info = ground_truth_gym_env.reset()
for _ in range(rollout_steps):
    actions = groundtruth_model.predict(obs)[0]
    for j in range(len(actions)):
        all_obs.append([c(info[j]['observation']) for c in concept_list])
        all_actions.append(actions[j])
    obs, _, _, _, info = ground_truth_gym_env.step(actions)

all_obs = np.asarray(all_obs, dtype=np.int8)
all_actions = np.asarray(all_actions)

N, K = all_obs.shape

# --------------------------------------------------
# Sample cross-action pairs
# --------------------------------------------------
idx_i = rng.integers(0, N, size=5 * num_pairs_lp)
idx_j = rng.integers(0, N, size=5 * num_pairs_lp)

valid = all_actions[idx_i] != all_actions[idx_j]
idx_i = idx_i[valid][:num_pairs_lp]
idx_j = idx_j[valid][:num_pairs_lp]

if len(idx_i) == 0:
    raise ValueError("No cross-action pairs sampled")

disagreement = (all_obs[idx_i] != all_obs[idx_j]).astype(np.int8)
M = disagreement.shape[0]

# --------------------------------------------------
# Precompute weights
# --------------------------------------------------
w = np.array([-math.log(max(1e-12, 1.0 - acc_list[d])) for d in range(K)])

# Identify used concepts (sparsity)
used_concepts = np.where(disagreement.sum(axis=0) > 0)[0]

# --------------------------------------------------
# PWL range estimation
# --------------------------------------------------
max_t = max(
    sum(w[d] for d in np.where(disagreement[p])[0])
    for p in range(M)
    if disagreement[p].any()
)
min_z = -max_t

if min_z < -10:
    breakpoints = [min_z, -8, -6, -4, -2, 0]
else:
    breakpoints = [min_z, min_z / 2, -2, -1, 0]

exp_vals = [math.exp(z) for z in breakpoints]


In [48]:
acc_list[6] = acc_list[7] = 0.01

In [52]:
model = gp.Model("stochastic_max_coverage")
model.Params.OutputFlag = 0
model.Params.Presolve = 2
model.Params.MIPFocus = 1
model.Params.Cuts = 2

vtype = GRB.BINARY
x = model.addVars(used_concepts, lb=0, ub=1, vtype=vtype, name="x")

z = model.addVars(M, lb=min_z, ub=0, vtype=GRB.CONTINUOUS, name="z")
u = model.addVars(M, lb=math.exp(min_z), ub=1.0, vtype=GRB.CONTINUOUS, name="u")
y = model.addVars(M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="y")

model.update()

# z_p = -sum_d A[p,d] w_d x_d
for p in range(M):
    D = np.where(disagreement[p])[0]
    if len(D) > 0:
        model.addConstr(
            z[p] == -gp.quicksum(w[d] * x[d] for d in D if d in x),
            name=f"zdef_{p}",
        )
    else:
        model.addConstr(z[p] == 0.0)

    model.addGenConstrPWL(z[p], u[p], breakpoints, exp_vals, name=f"exp_{p}")
    model.addConstr(y[p] == 1 - u[p], name=f"ydef_{p}")

# Budget
model.addConstr(gp.quicksum(x[d] for d in x) == num_concepts_selected)

# Objective
model.setObjective(gp.quicksum(y[p] for p in range(M)), GRB.MAXIMIZE)

model.optimize()

# --------------------------------------------------
# Extract solution
# --------------------------------------------------
if model.SolCount == 0:
    raise RuntimeError("No solution found")

x_vals = np.zeros(K)
for d in x:
    x_vals[d] = x[d].X

In [57]:
x_vals[[0,2,8]]

array([1., 1., 1.])

In [44]:
x_vals

array([-0., -0., -0.,  1., -0., -0.,  1.,  1., -0., -0.,  0., -0.])

In [10]:
subset_concepts, idx = multiple_lp_selection(ground_truth_gym_env,concept_list,num_concepts_selected,"q_value",q_estimates,"human_selected_binary",acc_list)

6
0
100
200
300


KeyboardInterrupt: 

In [9]:
subset_concept, idx = policy_coverage_selection_lp(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

There are 8000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
Coverage 0.9843


In [10]:
s = [1.0, 1.0, 1.0, 1.0, 0.9723301011417069, 0.942472000200446, 0.9930177979337359, 0.9836887073739069, 0.9942121219713863, 0.9798134181888034, 1.0, 1.0, 1.0, 1.0, 0.972614066297512, 0.942597278945654, 0.9930010941010415, 0.9837137631229486, 0.9942121219713863, 0.9797466028580257, 1.0, 1.0, 1.0, 1.0, 0.9725806586321232, 0.9429313555995423, 0.9929927421846942, 0.9836803554575597, 0.9942121219713863, 0.979796714356109, 1.0, 1.0, 1.0, 1.0, 0.972655825879248, 0.94243024061871, 0.9929760383519999, 0.9837054112066014, 0.9942121219713863, 0.9797883624397618, 1.0, 1.0, 1.0, 1.0, 0.9725890105484704, 0.9428812441014591, 0.9930261498500831, 0.9837471707883374, 0.9942121219713863, 0.9798050662724562, 1.0, 1.0, 1.0, 1.0, 0.9724553798869151, 0.9426891500254734, 0.9930512055991247, 0.9836887073739069, 0.9942121219713863, 0.9798050662724562, 1.0, 1.0, 1.0, 1.0, 0.9725639547994287, 0.942472000200446, 0.9930345017664303, 0.9836051882104349, 0.9942121219713863, 0.9796964913599425, 1.0, 1.0, 1.0, 1.0, 0.9725806586321232, 0.9426056308620013, 0.9930428536827776, 0.9836970592902542, 0.9942121219713863, 0.9798718816032339, 1.0, 1.0, 1.0, 1.0, 0.9728813276206225, 0.9423383695388907, 0.9930846132645136, 0.9837638746210318, 0.9942121219713863, 0.9796797875272482, 1.0, 1.0, 1.0, 1.0, 0.972856271871581, 0.9423717772042796, 0.9930094460173887, 0.9837221150392957, 0.9942121219713863, 0.9796547317782065, 1.0, 1.0, 1.0, 1.0, 0.9782683136645703, 0.9393233277375493, 0.9920656794701544, 0.984340156848989, 0.9942121219713863, 0.9817260070323136, 0.9890172300034242, 0.9878062021330795, 0.9984298397267253, 0.9916731394018358, 0.9780595157558902, 0.939306623904855, 0.992090735219196, 0.9843485087653362, 0.9942121219713863, 0.9817594146977023, 0.9926002021163756, 0.992775592359667, 0.9898941812198809, 0.9924999791202092, 0.9778423659308628, 0.9392398085740773, 0.9920656794701544, 0.9843735645143779, 0.9942121219713863, 0.9816007282871054, 0.9960913031495077, 0.9923747003750011, 0.9972939791035053, 0.9939699163973174, 0.9775834565240995, 0.9391646413269524, 0.992090735219196, 0.9843485087653362, 0.9942121219713863, 0.9815840244544111, 1.0, 1.0, 1.0, 1.0, 0.9780678676722374, 0.9394402545664102, 0.9920656794701544, 0.9843652125980307, 0.9942121219713863, 0.9816508397851887, 1.0, 1.0, 1.0, 1.0, 0.9781513868357095, 0.9393817911519798, 0.9920740313865016, 0.9843652125980307, 0.9942121219713863, 0.981742710865008, 1.0, 1.0, 1.0, 1.0, 0.9780678676722374, 0.939348383486591, 0.9920740313865016, 0.9844487317615027, 0.9942121219713863, 0.9817260070323136, 1.0, 1.0, 1.0, 1.0, 0.9778423659308628, 0.9391479374942581, 0.9920573275538073, 0.9844403798451554, 0.9942121219713863, 0.9816675436178831, 1.0, 1.0, 1.0, 1.0, 0.9779592927597237, 0.9390894740798276, 0.992090735219196, 0.9843318049326418, 0.9942121219713863, 0.981742710865008], [1.0, 1.0, 1.0, 1.0, 0.9775255599846556, 0.9254882665910569, 0.993853928648865, 0.97866804543256, 0.9999499641409677, 0.9775338992944944, 1.0, 1.0, 1.0, 1.0, 0.9775505779141718, 0.9256550527878313, 0.9938956251980586, 0.9785429557849792, 0.9999499641409677, 0.9775255599846556, 1.0, 1.0, 1.0, 1.0, 0.9774838634354621, 0.9253715162533149, 0.993853928648865, 0.9786180095735277, 0.9999499641409677, 0.9774922027453008, 1.0, 1.0, 1.0, 1.0, 0.9775005420551395, 0.9254966059008957, 0.9938622679587037, 0.9786597061227212, 0.9999499641409677, 0.9775088813649783, 1.0, 1.0, 1.0, 1.0, 0.9773671130977201, 0.9255549810697666, 0.9938205714095102, 0.9787847957703021, 0.9999499641409677, 0.9776589889420751, 1.0, 1.0, 1.0, 1.0, 0.9774671848157847, 0.9257301065763798, 0.993853928648865, 0.9784428840669146, 0.9999499641409677, 0.977458845505946, 1.0, 1.0, 1.0, 1.0, 0.9774338275764298, 0.9250546224794436, 0.9939039645078973, 0.9784512233767534, 0.9999499641409677, 0.9774004703370749, 1.0, 1.0, 1.0, 1.0, 0.9774838634354621, 0.9255799989992828, 0.9938872858882198, 0.9785846523341728, 0.9999499641409677, 0.9776423103223977, 1.0, 1.0, 1.0, 1.0, 0.9775422386043331, 0.925546641759928, 0.9938872858882198, 0.9784512233767534, 0.9999499641409677, 0.9776089530830429, 1.0, 1.0, 1.0, 1.0, 0.9774755241256233, 0.9253881948729923, 0.993853928648865, 0.9786180095735277, 0.9999499641409677, 0.9775505779141718, 1.0, 1.0, 1.0, 1.0, 0.9775088813649783, 0.9275814333605751, 0.9944710375769301, 0.9832630051536935, 0.9999499641409677, 0.9809446770185299, 0.9911269743316043, 0.9924195673566055, 0.9993745517620962, 0.9938289107193489, 0.977575595843688, 0.9274730223326717, 0.994446019647414, 0.9832963623930484, 0.9999499641409677, 0.9805944260053038, 0.9980402621879013, 0.9923528528778959, 0.9872825524959554, 0.9942792334506396, 0.9772420234501392, 0.9276564871491235, 0.9944543589572526, 0.9834297913504678, 0.9999499641409677, 0.9808195873709492, 1.0, 0.983246326534016, 0.9960471671364478, 0.9910269026135397, 0.9773504344780426, 0.9274563437129943, 0.994446019647414, 0.9832129692946612, 0.9999499641409677, 0.9807862301315943, 1.0, 1.0, 1.0, 1.0, 0.977517220674817, 0.9274980402621879, 0.9944626982670914, 0.983187951365145, 0.9999499641409677, 0.9808279266807879, 1.0, 1.0, 1.0, 1.0, 0.977575595843688, 0.9272478609670264, 0.9944210017178978, 0.9833130410127258, 0.9999499641409677, 0.9805777473856264, 1.0, 1.0, 1.0, 1.0, 0.9776256317027203, 0.9275313975015428, 0.9944793768867688, 0.9831796120553064, 0.9999499641409677, 0.9808029087512717, 1.0, 1.0, 1.0, 1.0, 0.9772920593091716, 0.9270560568407359, 0.9944543589572526, 0.9833380589422419, 0.9999499641409677, 0.9804860149774005, 1.0, 1.0, 1.0, 1.0, 0.9775922744633654, 0.9276314692196074, 0.9944710375769301, 0.9834214520406291, 0.9999499641409677, 0.9807278549627233], [1.0, 1.0, 1.0, 1.0, 0.9858337087650798, 0.9346832190352239, 0.9936176603093557, 0.9765563731624701, 0.9999415995060987, 0.9767148887887738, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9347833341676262, 0.9936593749478567, 0.9764646009577681, 0.9999415995060987, 0.9766898600056731, 1.0, 1.0, 1.0, 1.0, 0.9859254809697819, 0.934733276601425, 0.9936844037309572, 0.9764896297408686, 0.9999415995060987, 0.9767315746441741, 1.0, 1.0, 1.0, 1.0, 0.9859171380420817, 0.9345664180474212, 0.9936426890924563, 0.9766147736563715, 0.9999415995060987, 0.9767649463549749, 1.0, 1.0, 1.0, 1.0, 0.9858337087650798, 0.9347749912399259, 0.993626003237056, 0.9765146585239692, 0.9999415995060987, 0.9765647160901704, 1.0, 1.0, 1.0, 1.0, 0.9859421668251823, 0.9349585356493301, 0.9936844037309572, 0.9764062004638667, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9346581902521233, 0.9936009744539553, 0.9764228863192671, 0.9999415995060987, 0.976681517077973, 1.0, 1.0, 1.0, 1.0, 0.9859171380420817, 0.9346498473244231, 0.9936176603093557, 0.9765396873070697, 0.9999415995060987, 0.9767399175718743, 1.0, 1.0, 1.0, 1.0, 0.9859087951143816, 0.9347165907460246, 0.993626003237056, 0.9764896297408686, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9857836511988787, 0.9346748761075236, 0.9936510320201565, 0.9764812868131685, 0.9999415995060987, 0.9766231165840716, 1.0, 1.0, 1.0, 1.0, 0.9818457893243897, 0.9357761425639486, 0.9940848642605663, 0.9784418748227128, 0.9999582853614991, 0.9783667884734111, 0.9929085114548397, 0.9862425122223891, 0.9985983881463684, 0.9846573559593532, 0.9818624751797901, 0.9354507683836412, 0.9941599506098681, 0.9786170763044167, 0.9999582853614991, 0.9783501026180107, 0.9975722080392452, 0.9926415377684337, 0.9936009744539553, 0.9905724916987869, 0.9817873888304884, 0.9357177420700472, 0.9941349218267674, 0.9785336470274149, 0.9999582853614991, 0.9784168460396122, 1.0, 0.9939597203450635, 0.9975138075453438, 0.9943101233084715, 0.9818040746858888, 0.9359680299010529, 0.9940598354774658, 0.9785753616659159, 0.9999582853614991, 0.9784418748227128, 1.0, 1.0, 1.0, 1.0, 0.9818207605412892, 0.9357260849977475, 0.9941265788990673, 0.9785253040997147, 0.9999582853614991, 0.9784085031119121, 1.0, 1.0, 1.0, 1.0, 0.9819041898182911, 0.9357093991423471, 0.9940765213328662, 0.9785336470274149, 0.9999582853614991, 0.9784168460396122, 1.0, 1.0, 1.0, 1.0, 0.9818207605412892, 0.9357260849977475, 0.9941349218267674, 0.9785670187382156, 0.9999582853614991, 0.9783501026180107, 1.0, 1.0, 1.0, 1.0, 0.9817873888304884, 0.9358512289132502, 0.9941182359713671, 0.9785503328828152, 0.9999582853614991, 0.9784168460396122, 1.0, 1.0, 1.0, 1.0, 0.9819041898182911, 0.9357010562146468, 0.9941015501159667, 0.9784418748227128, 0.9999582853614991, 0.9784669036058133]

In [15]:
subset_concept, idx = policy_coverage_selection_exp_lp(
    ground_truth_gym_env,
    concept_list,s[0],
    num_concepts_selected,
    groundtruth_model)

48.0


In [19]:
rng = np.random.default_rng()
num_pairs_lp=20_000
rollout_steps=1000
# --------------------------------------------------
# Rollout data collection
# --------------------------------------------------
all_obs = []
all_actions = []

obs, info = ground_truth_gym_env.reset()
for _ in range(rollout_steps):
    actions = groundtruth_model.predict(obs)[0]
    for j in range(len(actions)):
        all_obs.append([c(info[j]['observation']) for c in concept_list])
        all_actions.append(actions[j])
    obs, _, _, _, info = ground_truth_gym_env.step(actions)

all_obs = np.asarray(all_obs, dtype=np.int8)
all_actions = np.asarray(all_actions)

N, K = all_obs.shape

# --------------------------------------------------
# Sample cross-action pairs
# --------------------------------------------------
idx_i = rng.integers(0, N, size=5 * num_pairs_lp)
idx_j = rng.integers(0, N, size=5 * num_pairs_lp)

valid = all_actions[idx_i] != all_actions[idx_j]
idx_i = idx_i[valid][:num_pairs_lp]
idx_j = idx_j[valid][:num_pairs_lp]

In [23]:
acc_list = s[0]
import math 

In [47]:
disagreement = (all_obs[idx_i] != all_obs[idx_j]).astype(np.int8)
M = disagreement.shape[0]

# --------------------------------------------------
# Precompute weights
# --------------------------------------------------
w = np.array([-math.log(max(1e-12, 1.0 - acc_list[d])) for d in range(K)])

# Identify used concepts (sparsity)
used_concepts = np.where(disagreement.sum(axis=0) > 0)[0]

# --------------------------------------------------
# PWL range estimation
# --------------------------------------------------
max_t = max(
    sum(w[d] for d in np.where(disagreement[p])[0])
    for p in range(M)
    if disagreement[p].any()
)
min_z = -max_t

if min_z < -10:
    breakpoints = [min_z, -2, -1,-0.5,0]
else:
    breakpoints = [min_z, min_z / 2, -2, -1, 0]
exp_vals = [math.exp(z) for z in breakpoints]


In [48]:
model = gp.Model("stochastic_max_coverage")
model.Params.OutputFlag = 0
model.Params.Presolve = 2
model.Params.MIPFocus = 1
model.Params.Cuts = 2

vtype = GRB.BINARY
x = model.addVars(used_concepts, lb=0, ub=1, vtype=vtype, name="x")

z = model.addVars(M, lb=min_z, ub=0, vtype=GRB.CONTINUOUS, name="z")
u = model.addVars(M, lb=math.exp(min_z), ub=1.0, vtype=GRB.CONTINUOUS, name="u")
y = model.addVars(M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="y")

model.update()

# z_p = -sum_d A[p,d] w_d x_d
for p in range(M):
    D = np.where(disagreement[p])[0]
    if len(D) > 0:
        model.addConstr(
            z[p] == -gp.quicksum(w[d] * x[d] for d in D if d in x),
            name=f"zdef_{p}",
        )
    else:
        model.addConstr(z[p] == 0.0)

    model.addGenConstrPWL(z[p], u[p], breakpoints, exp_vals, name=f"exp_{p}")
    model.addConstr(y[p] == 1 - u[p], name=f"ydef_{p}")

# Budget
model.addConstr(gp.quicksum(x[d] for d in x) == num_concepts_selected)

# Objective
model.setObjective(gp.quicksum(y[p] for p in range(M)), GRB.MAXIMIZE)

model.optimize()
x_vals = np.zeros(K)
for d in x:
    x_vals[d] = x[d].X
idx = [i for i in range(len(x_vals)) if x_vals[i] > 0.5]

In [49]:
idx 

[6,
 7,
 16,
 17,
 25,
 26,
 27,
 36,
 37,
 46,
 47,
 56,
 57,
 66,
 67,
 76,
 77,
 79,
 84,
 86,
 87,
 96,
 97,
 104,
 106,
 107,
 110,
 111,
 113,
 116,
 117,
 122,
 123,
 126,
 127,
 136,
 137,
 145,
 146,
 147,
 156,
 157,
 166,
 167,
 176,
 177,
 186,
 187]

In [25]:
exp_vals

[1.1364128085613893e-174,
 0.00033546262790251185,
 0.0024787521766663585,
 0.01831563888873418,
 0.1353352832366127,
 1.0]

In [482]:
len(idx)

48

In [483]:
subset_concept, idx = policy_coverage_selection_lp_advantage(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

Starting stable training for sparse rewards...
Total of 200000 steps
Step 0/200000, Loss mean: 0.0000
Step 5000/200000, Loss mean: 0.8814
Step 10000/200000, Loss mean: 0.7551
Step 15000/200000, Loss mean: 0.8678
Step 20000/200000, Loss mean: 0.5986
Updated target network at step 500, Avg recent loss: 0.5936
Step 25000/200000, Loss mean: 0.2069
Step 30000/200000, Loss mean: 0.1720
Step 35000/200000, Loss mean: 0.1701
Step 40000/200000, Loss mean: 0.1706
Updated target network at step 1000, Avg recent loss: 0.1706
Step 45000/200000, Loss mean: 0.0824
Step 50000/200000, Loss mean: 0.0754
Step 55000/200000, Loss mean: 0.0768
Step 60000/200000, Loss mean: 0.0919
Updated target network at step 1500, Avg recent loss: 0.0920
Step 65000/200000, Loss mean: 0.0813
Step 70000/200000, Loss mean: 0.0909
Episode 25, Avg Reward: 70.12, Loss (mean/std/max): 0.09/0.05/0.21, Epsilon: 0.041
Step 75000/200000, Loss mean: 0.0841
Step 80000/200000, Loss mean: 0.0887
Updated target network at step 2000, Avg r

In [484]:
len(idx)

48